In [32]:
from google.colab import files
subido = files.upload()
!unzip -q -o proyecto_tiktok_AE3.zip
%cd repo/AE3
!pip install -q python-docx

Saving proyecto_tiktok_AE3.zip to proyecto_tiktok_AE3.zip
/content/repo/AE3/repo/AE3/repo/AE3


# AE3 – Transformación y enriquecimiento de datos (proyecto TikTok)
**Integrante(s) / Grupo / Docente:** _completar_

Este notebook continúa el proyecto de AE2 (mismo dataset, misma necesidad y mismo repositorio). Flujo:

```
Dataset de AE2 (HFDatasetLoader) → transformación y validación → enriquecimiento
      → respuestas a las preguntas de negocio (tablas) → exportación (.csv, .txt) → documento
```
> Ejecutar con *Run All* desde la carpeta `AE3/`. Todas las rutas son relativas. El notebook **conserva** una copia sin modificar (`df_original`) para comparar antes y después.

## 1. Necesidad, preguntas de negocio y variables

**Necesidad (AE2, escenario hipotético):** el equipo de contenidos de una agencia que gestiona cuentas de marcas en TikTok necesita conocer cómo se comportan el alcance (vistas) y la interacción de las publicaciones según su formato y su carácter comercial, para orientar qué producir. Es una necesidad *propuesta*, no un problema observado y comprobado.

**Pregunta de AE2:** ¿cómo varían las vistas medianas y la tasa de interacción según el tipo de publicación, el rango de duración, el país y si son publicidad o contenido de TikTok Shop?

Para poder responderla con tablas, se desglosa en seis preguntas de negocio:

| N.º | Pregunta de negocio | Variables necesarias | Tabla de salida | Resultado esperado |
|---|---|---|---|---|
| P1 | ¿Cómo se distribuyen las vistas y la interacción en la muestra? | `views`, `interacciones_totales`, `tasa_interaccion`, `rango_vistas` | `p1a_resumen_general`, `p1b_rango_vistas` | Mediana, cuartiles y concentración de vistas |
| P2 | ¿Cómo varían según el tipo (video o foto)? | `type`, `views`, `tasa_interaccion` | `p2_tipo_publicacion` | Mediana de vistas e interacción por tipo |
| P3 | ¿Cómo varían según el rango de duración? | `duration_ms` → `rango_duracion` | `p3_rango_duracion` | Mediana de vistas e interacción por rango |
| P4 | ¿Cómo varían según el país? | `country` → `pais_agrupado` | `p4_pais`, `p4b_pais_completo` | Comparación entre los países con más publicaciones |
| P5 | ¿Cómo varían según el carácter comercial (publicidad, TikTok Shop, contenido de marca)? | `is_ad`, `is_shop_video`, `branded_content` → `categoria_comercial` | `p5a_categoria_comercial`, `p5b_indicadores_comerciales` | Mediana de vistas e interacción por categoría |
| P6 | ¿Cómo cambian según el año de publicación? (pregunta de contexto) | `posted_at` → `anio_publicacion`, `edad_dias`, `vistas_por_dia` | `p6_anio_publicacion` | Efecto del tiempo en línea sobre las métricas |

### Ajustes respecto a AE2 y su justificación
- **A1. Desglose en P1–P6.** Una sola pregunta con cuatro dimensiones no cabe en una tabla; se separa por dimensión y se añade una línea base (P1).
- **A2. P6 es nueva.** AE2 advirtió que cada registro tiene métricas medidas en fechas distintas; sin el año de publicación y el tiempo en línea, las comparaciones de P2–P5 podrían confundirse con la antigüedad.
- **A3. País agrupado.** Se comparan los 10 países con más publicaciones; el resto se agrupa en «Otros países» y los vacíos en «SIN DATO», para que los totales cuadren. La tabla completa se exporta aparte.
- **A4. Duración solo en videos.** Según la ficha, las fotos tienen duración 0; se clasifican aparte y no entran en los rankings de P3.
- **A5. Tasa agregada.** Además de la mediana de la tasa por publicación se calcula la tasa agregada (suma de interacciones / suma de vistas) para no depender de publicaciones pequeñas.
- **A6. Alcance de variables.** De las 27 columnas cargadas en AE2 se conservan las que responden las preguntas; no se incorpora ninguna fuente externa porque las variables existentes bastan.

## 2. Configuración y carga del dataset de AE2

In [33]:
import json
import textwrap
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.options.display.float_format = "{:,.2f}".format

# ---- Dataset y muestra: mismos parámetros que AE2 ----
DATASET_ID = "datasocial/tiktok-5.6B-videos"
SPLIT = "train"
ENLACE = f"https://huggingface.co/datasets/{DATASET_ID}"
TOTAL_FILAS_DATASET = 5_598_141_717        # según la ficha
N_FILAS, SEED = 200_000, 42

COLUMNAS_CARGADAS = ["video_id","posted_at","country","language","type","duration_ms","image_count",
    "width","height","hashtags","sound_id","is_ad","branded_content","product_id","is_shop_video",
    "seller_id","is_ai_generated","not_recommended_for_fyp","sound_muted","views","likes","comments",
    "shares","saves","downloads","engagement_rate","stats_updated_at"]

# ---- Rutas relativas de salida ----
RUTA_PREPARADO = Path("datos/preparado")
RUTA_TABLAS = Path("datos/tablas")
RUTA_CALIDAD = Path("datos/calidad")
RUTA_RESULTADOS = Path("resultados")
for ruta in (RUTA_PREPARADO, RUTA_TABLAS, RUTA_CALIDAD, RUTA_RESULTADOS):
    ruta.mkdir(parents=True, exist_ok=True)

# ---- Reglas del análisis ----
TOP_PAISES = 10        # países que se comparan individualmente
MIN_N_INTERP = 100     # mínimo de registros para incluir un grupo en rankings de la interpretación
TOL_TASA = 1e-3        # tolerancia al validar engagement_rate contra la fórmula

# ---- Formato de números en español para los textos interpretativos ----
def f_int(x):
    return f"{x:,.0f}".replace(",", ".")
def f_dec(x, d=1):
    return f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
def f_pct(x, d=2):
    return f_dec(x, d) + " %"

In [34]:
loader = HFDatasetLoader(DATASET_ID, split=SPLIT, columns=COLUMNAS_CARGADAS, max_rows=N_FILAS, seed=SEED)
df_original = loader.load()          # versión SIN modificar, para comparar al final
df = df_original.copy()              # sobre esta copia se trabaja

FECHA_CONSULTA = pd.Timestamp.today().date().isoformat()
print(loader)
print("Revisión (commit):", loader.revision)
print("Muestra:", loader.sample_note)
print(f"Registros cargados: {len(df_original):,} | columnas: {df_original.shape[1]}")

HFDatasetLoader(dataset_id='datasocial/tiktok-5.6B-videos', split='train', max_rows=200000)
Revisión (commit): a466d03349b4afa490ac8906c8b5d17f6006abc0
Muestra: Archivo 'data/2016-11.parquet' elegido al azar (semilla 42) entre 148 archivos parquet; 1 grupos de filas repartidos de 1; hasta 200,000 filas. No es una muestra aleatoria simple del dataset completo.
Registros cargados: 200,000 | columnas: 27


In [35]:
df_original.head()

,video_id,posted_at,country,language,type,duration_ms,image_count,width,height,hashtags,sound_id,is_ad,branded_content,product_id,is_shop_video,seller_id,is_ai_generated,not_recommended_for_fyp,sound_muted,views,likes,comments,shares,saves,downloads,engagement_rate,stats_updated_at
0,157771620990803968,2016-11-01 00:00:01+00:00,,un,video,5000,0,0,0,[],131820703,0,,0,0,0,NaN,0.00,0,60,2,1,0,0,0,0.05,2026-06-19 19:13:34+00:00
1,157771636836843520,2016-11-01 00:00:04+00:00,,un,video,7000,0,0,0,[],157581184582070272,0,,0,0,0,NaN,0.00,0,38,3,0,0,0,0,0.08,2026-06-14 22:27:44+00:00
2,157771643619196928,2016-11-01 00:00:06+00:00,,un,photo,0,0,0,0,"[happyhalloween, like4like, followforfollow]",157771643707187200,0,,0,0,0,NaN,0.00,0,104,23,0,0,0,0,0.22,2026-06-21 12:16:54+00:00
3,157771674799648768,2016-11-01 00:00:13+00:00,,un,photo,0,0,0,0,[],148185334,0,,0,0,0,NaN,0.00,0,87,15,0,3,0,0,0.21,2026-06-15 02:05:40+00:00
4,157771679375466496,2016-11-01 00:00:15+00:00,,un,video,9000,0,0,0,[],157771679513997312,0,,0,0,0,NaN,0.00,0,68,2,0,1,1,0,0.06,2026-06-15 09:23:57+00:00


## 3. Perfil inicial (antes de transformar)
Se guarda el perfil de `df_original` (tipos y nulos por columna) para la comparación posterior.

In [36]:
def perfil(d):
    return pd.DataFrame({"tipo": d.dtypes.astype(str), "nulos": d.isna().sum()})

def familia_tipo(dtype):
    if pd.api.types.is_bool_dtype(dtype):
        return "booleana"
    if pd.api.types.is_datetime64_any_dtype(dtype):
        return "fecha"
    if pd.api.types.is_numeric_dtype(dtype):
        return "numérica"
    return "texto / lista / categoría"

perfil_antes = perfil(df_original)
perfil_antes

,tipo,nulos
video_id,uint64,0
posted_at,"datetime64[ms, UTC]",0
country,object,0
language,object,0
type,object,0
duration_ms,uint32,0
image_count,uint8,0
width,uint16,0
height,uint16,0
hashtags,object,0


## 4. Transformación y validación
Cada paso se registra en una **bitácora** (qué se hizo, a cuántos registros afectó y por qué). **No se elimina ningún registro** salvo los duplicados exactos por `video_id`, si existieran; los registros dudosos se *marcan* con indicadores `f_*` y se excluyen solo de los cálculos que no pueden usarlos.

In [37]:
bitacora = []
def registrar(paso, operacion, columnas, afectados, justificacion):
    bitacora.append({"paso": paso, "operacion": operacion, "columnas": columnas,
                     "registros_afectados": int(afectados), "justificacion": justificacion})
    print(f"[{paso}] {operacion}: {int(afectados):,} registros afectados")

### 4.1 Corregir tipos y formatos

In [38]:
COLS_CONTEO = ["duration_ms", "views", "likes", "comments", "shares", "saves"]
BANDERAS = ["is_ad", "is_shop_video"]

# Conteos: enteros sin signo (uint64) -> enteros con signo y nulos permitidos (Int64)
for c in COLS_CONTEO:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
df["engagement_rate"] = pd.to_numeric(df["engagement_rate"], errors="coerce").astype("float64")

# Fechas: se fuerzan a UTC; los valores no interpretables quedarían como nulos
nulos_fecha_antes = int(df["posted_at"].isna().sum() + df["stats_updated_at"].isna().sum())
df["posted_at"] = pd.to_datetime(df["posted_at"], utc=True, errors="coerce")
df["stats_updated_at"] = pd.to_datetime(df["stats_updated_at"], utc=True, errors="coerce")
fechas_invalidas = int(df["posted_at"].isna().sum() + df["stats_updated_at"].isna().sum()) - nulos_fecha_antes

# Indicadores 0/1 -> booleanos (el nulo se conserva: significa "no indicado")
fuera_de_rango = 0
for c in BANDERAS:
    num = pd.to_numeric(df[c], errors="coerce")
    fuera_de_rango += int((~num.isin([0, 1]) & num.notna()).sum())
    df[c] = num.map({0: False, 1: True}).astype("boolean")

print("Fechas no interpretables (convertidas a nulo):", fechas_invalidas)
print("Valores de is_ad / is_shop_video distintos de 0, 1 o nulo:", fuera_de_rango)
registrar("T1", "Corrección de tipos y formatos",
          ", ".join(COLS_CONTEO + ["engagement_rate", "posted_at", "stats_updated_at"] + BANDERAS),
          len(df),
          "Los conteos llegan como uint64: se pasan a Int64 (con signo) para evitar desbordamientos al restar "
          "y admitir nulos; las fechas se fuerzan a UTC; los indicadores 0/1 pasan a booleano. "
          f"Fechas no interpretables: {fechas_invalidas}; valores fuera de 0/1: {fuera_de_rango}.")
df.dtypes

Fechas no interpretables (convertidas a nulo): 0
Valores de is_ad / is_shop_video distintos de 0, 1 o nulo: 0
[T1] Corrección de tipos y formatos: 200,000 registros afectados


,0
video_id,uint64
posted_at,"datetime64[ms, UTC]"
country,object
language,object
type,object
duration_ms,Int64
image_count,uint8
width,uint16
height,uint16
hashtags,object


### 4.2 Duplicados

In [39]:
n_ids_nulos = int(df["video_id"].isna().sum())
n_dup_id = int(df.duplicated(subset="video_id", keep="first").sum())
n_dup_fila = int(df.drop(columns=["hashtags"]).duplicated().sum())
print(f"video_id nulos: {n_ids_nulos} | duplicados por video_id: {n_dup_id} | filas totalmente duplicadas: {n_dup_fila}")

filas_antes_dup = len(df)
if n_dup_id > 0:
    # Criterio: se conserva el registro con métricas más recientes (stats_updated_at)
    df = df.sort_values("stats_updated_at").drop_duplicates(subset="video_id", keep="last").reset_index(drop=True)
registrar("T2", "Revisión y tratamiento de duplicados", "video_id", n_dup_id,
          f"video_id identifica una publicación. Se detectaron {n_dup_id} duplicados por video_id y {n_dup_fila} filas "
          "totalmente duplicadas. " + ("Se conservó el registro con stats_updated_at más reciente."
                                       if n_dup_id else "No se eliminó ningún registro."))
print(f"Filas antes: {filas_antes_dup:,} | después: {len(df):,}")

video_id nulos: 0 | duplicados por video_id: 0 | filas totalmente duplicadas: 0
[T2] Revisión y tratamiento de duplicados: 0 registros afectados
Filas antes: 200,000 | después: 200,000


### 4.3 Normalizar categorías

In [40]:
# type: minúsculas y sin espacios
tipo_previo = df["type"].astype("string")
df["type"] = tipo_previo.str.strip().str.lower()
cambios_tipo = int((tipo_previo.fillna("") != df["type"].fillna("")).sum())

# country: mayúsculas; el texto vacío significa "sin dato" y se etiqueta explícitamente
pais_previo = df["country"].astype("string")
df["country"] = pais_previo.fillna("").str.strip().str.upper()
paises_vacios = int((df["country"] == "").sum())
cambios_pais = int(((pais_previo.fillna("") != df["country"]) & (df["country"] != "")).sum())
df.loc[df["country"] == "", "country"] = "SIN DATO"

# branded_content: minúsculas; vacío significa que no hay contenido de marca
marca_previa = df["branded_content"].astype("string")
df["branded_content"] = marca_previa.fillna("").str.strip().str.lower()
marcas_vacias = int((df["branded_content"] == "").sum())
cambios_marca = int(((marca_previa.fillna("") != df["branded_content"]) & (df["branded_content"] != "")).sum())
df.loc[df["branded_content"] == "", "branded_content"] = "ninguno"

tipos_no_reconocidos = int((~df["type"].isin(["video", "photo"])).sum())
print("type -> valores:", df["type"].value_counts(dropna=False).to_dict())
print("branded_content -> valores:", df["branded_content"].value_counts(dropna=False).to_dict())
print(f"country vacío -> 'SIN DATO': {paises_vacios:,} ({paises_vacios/len(df):.1%})")
registrar("T3", "Normalización de categorías", "type, country, branded_content",
          cambios_tipo + paises_vacios + cambios_pais + marcas_vacias + cambios_marca,
          f"Texto sin espacios y con mayúsculas/minúsculas uniformes. country vacío -> 'SIN DATO' ({paises_vacios:,}); "
          f"branded_content vacío -> 'ninguno' ({marcas_vacias:,}). Registros con type distinto de video/photo: {tipos_no_reconocidos}.")

type -> valores: {'video': 160456, 'photo': 39544}
branded_content -> valores: {'ninguno': 200000}
country vacío -> 'SIN DATO': 200,000 (100.0%)
[T3] Normalización de categorías: 400,000 registros afectados


### 4.4 Controles de calidad e inconsistencias
Cada control se **informa aunque no detecte problemas**. Tratamiento: los registros se marcan con `f_*`; solo se excluyen de los cálculos que dependen del campo dudoso.

In [41]:
n = len(df)
views_f = df["views"].astype("float64").to_numpy()
likes_f = df["likes"].astype("float64").to_numpy()
dur_f = df["duration_ms"].astype("float64").to_numpy()
es_video = (df["type"] == "video").fillna(False).to_numpy(bool)
es_foto = (df["type"] == "photo").fillna(False).to_numpy(bool)

f_fecha = (df["posted_at"].isna() | df["stats_updated_at"].isna()
           | (df["posted_at"] > df["stats_updated_at"])).to_numpy(bool)
f_vistas_nulas = np.isnan(views_f)
f_sin_vistas = (views_f == 0)
f_likes = (likes_f > views_f) & (views_f > 0)
f_dur = (es_foto & (dur_f > 0)) | (es_video & (dur_f == 0))
f_tipo = ~(es_video | es_foto)

df["f_fecha_incoherente"] = f_fecha
df["f_sin_vistas"] = f_sin_vistas
df["f_likes_mayor_vistas"] = f_likes
df["f_duracion_inconsistente"] = f_dur
df["f_tipo_no_reconocido"] = f_tipo

# Registros que entran en los cálculos
df["analisis_vistas"] = ~f_fecha & ~f_vistas_nulas

controles = []
def control(nombre, criterio, mascara, tratamiento):
    k = int(np.asarray(mascara).sum())
    controles.append({"control": nombre, "criterio": criterio, "registros_detectados": k,
                      "porcentaje": round(100 * k / n, 4), "tratamiento": tratamiento})

control("Fechas incoherentes", "posted_at nulo, stats_updated_at nulo o publicación posterior a la medición",
        f_fecha, "Se excluye de todos los cálculos (no se puede confiar en la cronología)")
control("Vistas nulas", "views sin valor", f_vistas_nulas, "Se excluye de todos los cálculos")
control("Publicaciones sin vistas", "views = 0 (la tasa de interacción no está definida)",
        f_sin_vistas, "Se conserva para vistas; se excluye de los cálculos de tasa")
control("Me gusta mayores que vistas", "likes > views con views > 0",
        f_likes, "Se conserva para vistas; se excluye de los cálculos de tasa")
control("Duración inconsistente", "foto con duración > 0 o video con duración = 0",
        f_dur, "Se conserva; los videos sin duración van a una categoría propia en P3")
control("Tipo no reconocido", "type distinto de video o photo", f_tipo, "Se conserva y se informa")
control("Indicadores sin dato", "is_ad o is_shop_video nulos (TikTok no los indicó)",
        (df["is_ad"].isna() | df["is_shop_video"].isna()).to_numpy(bool),
        "Se tratan como 'no' al clasificar; se informa su cantidad")
control("Duplicados por video_id", "video_id repetido", np.zeros(n, bool),
        f"Detectados antes de limpiar: {n_dup_id}; " + ("se conservó el más reciente" if n_dup_id else "ninguno que tratar"))
controles[-1]["registros_detectados"] = n_dup_id
controles[-1]["porcentaje"] = round(100 * n_dup_id / (n + n_dup_id), 4)

tabla_controles = pd.DataFrame(controles)
registrar("T4", "Controles de calidad y marcado de registros", "f_* , analisis_vistas",
          int((f_fecha | f_vistas_nulas | f_sin_vistas | f_likes | f_dur | f_tipo).sum()),
          "Registros con al menos una marca; ninguno se elimina. Las exclusiones se aplican solo en los cálculos que dependen del campo dudoso.")
tabla_controles

[T4] Controles de calidad y marcado de registros: 16,401 registros afectados


,control,criterio,registros_detectados,porcentaje,tratamiento
0,Fechas incoherentes,"posted_at nulo, stats_updated_at nulo o public...",0,0.00,Se excluye de todos los cálculos (no se puede ...
1,Vistas nulas,views sin valor,0,0.00,Se excluye de todos los cálculos
2,Publicaciones sin vistas,views = 0 (la tasa de interacción no está defi...,1574,0.79,Se conserva para vistas; se excluye de los cál...
3,Me gusta mayores que vistas,likes > views con views > 0,14643,7.32,Se conserva para vistas; se excluye de los cál...
4,Duración inconsistente,foto con duración > 0 o video con duración = 0,210,0.10,Se conserva; los videos sin duración van a una...
5,Tipo no reconocido,type distinto de video o photo,0,0.00,Se conserva y se informa
6,Indicadores sin dato,is_ad o is_shop_video nulos (TikTok no los ind...,0,0.00,Se tratan como 'no' al clasificar; se informa ...
7,Duplicados por video_id,video_id repetido,0,0.00,Detectados antes de limpiar: 0; ninguno que tr...


## 5. Enriquecimiento
Se incorporan variables calculadas con los datos existentes (no hace falta otra fuente). Cada una se documenta con su regla, significado y utilidad.

In [42]:
derivadas = []
def derivar(nombre, regla, significado, utilidad):
    derivadas.append({"variable": nombre, "regla": regla, "significado": significado, "utilidad": utilidad})

# --- Periodo ---
df["anio_publicacion"] = df["posted_at"].dt.year.astype("Int64")
derivar("anio_publicacion", "año de posted_at (UTC)", "Año en que se publicó", "P6: comparar por periodo")

# --- Duración ---
views_f = df["views"].astype("float64").to_numpy()
dur_f = df["duration_ms"].astype("float64").to_numpy()
es_video = (df["type"] == "video").fillna(False).to_numpy(bool)
es_foto = (df["type"] == "photo").fillna(False).to_numpy(bool)

df["duracion_s"] = np.where(es_video & (dur_f > 0), dur_f / 1000, np.nan)
derivar("duracion_s", "duration_ms / 1000, solo videos con duración > 0; vacío en otros casos",
        "Duración en segundos", "P3: base para los rangos de duración")

ORDEN_DURACION = ["0–15 s", "15–30 s", "30–60 s", "1–3 min", "Más de 3 min",
                  "Video sin duración registrada", "Foto (sin duración)"]
cortes = pd.cut(df["duracion_s"], bins=[0, 15, 30, 60, 180, np.inf], labels=ORDEN_DURACION[:5], right=True)
df["rango_duracion"] = cortes.astype("object")
df.loc[es_video & df["duracion_s"].isna().to_numpy(bool), "rango_duracion"] = ORDEN_DURACION[5]
df.loc[es_foto, "rango_duracion"] = ORDEN_DURACION[6]
df["rango_duracion"] = df["rango_duracion"].fillna("Tipo no reconocido")
derivar("rango_duracion", "cortes (0,15], (15,30], (30,60], (60,180], >180 s; fotos y videos sin duración en categorías propias",
        "Rango de duración del video", "P3: comparar por duración")

# --- Alcance ---
ORDEN_VISTAS = ["Menos de 1 mil", "1 mil – 10 mil", "10 mil – 100 mil", "100 mil – 1 millón", "1 millón o más"]
df["rango_vistas"] = pd.cut(views_f, bins=[0, 1e3, 1e4, 1e5, 1e6, np.inf],
                            labels=ORDEN_VISTAS, right=False).astype("object")
df["rango_vistas"] = df["rango_vistas"].fillna("Sin dato")
derivar("rango_vistas", "cortes [0,1 mil), [1 mil,10 mil), [10 mil,100 mil), [100 mil,1 millón), ≥1 millón vistas",
        "Rango de alcance de la publicación", "P1: distribución del alcance; gráficas de AE4")

# --- Interacción ---
df["interacciones_totales"] = df["likes"] + df["comments"] + df["shares"] + df["saves"]
inter_f = df["interacciones_totales"].astype("float64").to_numpy()
derivar("interacciones_totales", "likes + comments + shares + saves", "Total de interacciones", "Numerador de la tasa de interacción")

df["tasa_interaccion_calc"] = np.where(views_f > 0, inter_f / views_f, np.nan)
derivar("tasa_interaccion_calc", "interacciones_totales / views, solo si views > 0",
        "Tasa recalculada con la fórmula de la ficha", "Variable intermedia: valida engagement_rate y completa faltantes (no se exporta)")

df["tasa_interaccion"] = df["engagement_rate"].fillna(df["tasa_interaccion_calc"])
df["tasa_imputada"] = (df["engagement_rate"].isna() & df["tasa_interaccion_calc"].notna()).to_numpy(bool)
derivar("tasa_interaccion", "engagement_rate; si es nulo, tasa_interaccion_calc", "Tasa de interacción usada en el análisis (proporción 0–1)", "P1–P6: métrica principal de interacción")
derivar("tasa_imputada", "True si engagement_rate era nulo y se completó con la fórmula", "Marca de tasa completada", "Trazabilidad del criterio de imputación")

# --- Carácter comercial ---
ad = df["is_ad"].fillna(False).astype(bool).to_numpy()
shop = df["is_shop_video"].fillna(False).astype(bool).to_numpy()
marca = (df["branded_content"] != "ninguno").to_numpy(bool)
CAT_AD, CAT_SHOP, CAT_MARCA, CAT_NINGUNA = ("Publicidad (is_ad)", "TikTok Shop (is_shop_video)",
                                            "Contenido de marca (branded_content)", "Sin señal comercial")
df["n_senales_comerciales"] = ad.astype(int) + shop.astype(int) + marca.astype(int)
df["categoria_comercial"] = np.select([ad, shop, marca], [CAT_AD, CAT_SHOP, CAT_MARCA], default=CAT_NINGUNA)
derivar("n_senales_comerciales", "is_ad + is_shop_video + (branded_content ≠ ninguno)", "Cuántas señales comerciales tiene", "Medir solapamientos entre categorías")
derivar("categoria_comercial", "prioridad: is_ad > is_shop_video > branded_content > ninguna", "Categoría comercial excluyente", "P5: comparar por carácter comercial")

# --- Otros indicadores ---
df["n_hashtags"] = df["hashtags"].map(lambda v: len(v) if hasattr(v, "__len__") else 0).astype("int64")
derivar("n_hashtags", "cantidad de elementos de la lista hashtags", "Número de hashtags", "Describir el uso de etiquetas; reemplaza a la lista (no exportable a CSV)")

f_fecha = df["f_fecha_incoherente"].to_numpy(bool)
edad = (df["stats_updated_at"] - df["posted_at"]).dt.days.astype("float64").to_numpy()
edad = np.where(f_fecha, np.nan, edad)
df["edad_dias"] = pd.array(edad, dtype="Float64").astype("Int64")
df["vistas_por_dia"] = np.where(np.isnan(edad), np.nan, views_f / np.maximum(edad, 1))
derivar("edad_dias", "días entre posted_at y stats_updated_at", "Tiempo en línea al medir las métricas", "P6: controlar el efecto de la antigüedad")
derivar("vistas_por_dia", "views / max(edad_dias, 1)", "Vistas promedio por día en línea", "P6: comparar publicaciones de distinta antigüedad")

# --- Países agrupados (los TOP_PAISES con más publicaciones analizables) ---
base_paises = df[df["analisis_vistas"] & (df["country"] != "SIN DATO")]
top_paises = base_paises["country"].value_counts().head(TOP_PAISES).index.tolist()
df["pais_agrupado"] = np.where(df["country"] == "SIN DATO", "SIN DATO",
                               np.where(df["country"].isin(top_paises), df["country"], "Otros países"))
derivar("pais_agrupado", f"los {TOP_PAISES} países con más publicaciones; resto = 'Otros países'; vacío = 'SIN DATO'",
        "País agrupado para comparar", "P4: comparar países con base suficiente")

# --- Máscara de análisis de tasa ---
df["analisis_tasa"] = (df["analisis_vistas"] & ~df["f_sin_vistas"] & ~df["f_likes_mayor_vistas"]
                       & df["tasa_interaccion"].notna()).to_numpy(bool)
derivar("analisis_vistas / analisis_tasa", "ver sección 4.4: sin fechas incoherentes ni vistas nulas; para la tasa, además views > 0, likes ≤ views y tasa disponible",
        "Indicadores de inclusión en los cálculos", "Garantizan que cada tabla use los mismos registros")
print(f"Variables derivadas: {len(derivadas)} | países comparados: {top_paises}")

/tmp/ipykernel_628/281512478.py:42: RuntimeWarning: divide by zero encountered in divide
  df["tasa_interaccion_calc"] = np.where(views_f > 0, inter_f / views_f, np.nan)
/tmp/ipykernel_628/281512478.py:42: RuntimeWarning: invalid value encountered in divide
  df["tasa_interaccion_calc"] = np.where(views_f > 0, inter_f / views_f, np.nan)


Variables derivadas: 15 | países comparados: []


### 5.1 Validación del enriquecimiento
Se comprueba que `engagement_rate` (dato original) coincida con la fórmula de la ficha, que las categorías sean excluyentes y que ningún registro quede sin clasificar.

In [43]:
valid = []
def validar(nombre, detalle, ok):
    valid.append({"verificacion": nombre, "detalle": detalle, "resultado": "OK" if ok else "REVISAR"})

comparables = df["engagement_rate"].notna() & df["tasa_interaccion_calc"].notna()
dif = (df.loc[comparables, "engagement_rate"] - df.loc[comparables, "tasa_interaccion_calc"]).abs()
n_dif = int((dif > TOL_TASA).sum())
validar("engagement_rate vs fórmula", f"{int(comparables.sum()):,} registros comparables; {n_dif:,} con diferencia > {TOL_TASA} (máx. {dif.max():.6f})", n_dif == 0)
validar("Tasas imputadas", f"{int(df['tasa_imputada'].sum()):,} registros con engagement_rate nulo completados con la fórmula", True)
validar("Categorías comerciales", f"{int((df['n_senales_comerciales'] >= 2).sum()):,} registros con 2 o más señales (se asignan por prioridad)", df["categoria_comercial"].notna().all())
validar("Rango de duración completo", "todo registro tiene rango_duracion", df["rango_duracion"].notna().all())
validar("Rango de vistas completo", "todo registro tiene rango_vistas", df["rango_vistas"].notna().all())
validar("Interacciones no negativas", "interacciones_totales >= 0", bool((df["interacciones_totales"].dropna() >= 0).all()))
validar("Países agrupados", "la suma de grupos de pais_agrupado cubre todos los registros", int(df["pais_agrupado"].value_counts().sum()) == len(df))
pd.DataFrame(valid)

,verificacion,detalle,resultado
0,engagement_rate vs fórmula,"198,426 registros comparables; 0 con diferenci...",OK
1,Tasas imputadas,0 registros con engagement_rate nulo completad...,OK
2,Categorías comerciales,0 registros con 2 o más señales (se asignan po...,OK
3,Rango de duración completo,todo registro tiene rango_duracion,OK
4,Rango de vistas completo,todo registro tiene rango_vistas,OK
5,Interacciones no negativas,interacciones_totales >= 0,OK
6,Países agrupados,la suma de grupos de pais_agrupado cubre todos...,OK


## 6. Selección de columnas y comparación antes / después

In [44]:
COLS_FINALES = [
    # identificación y fechas
    "video_id", "posted_at", "anio_publicacion", "edad_dias",
    # variables originales usadas
    "type", "duration_ms", "country", "is_ad", "is_shop_video", "branded_content",
    "views", "likes", "comments", "shares", "saves", "engagement_rate",
    # variables derivadas
    "duracion_s", "rango_duracion", "rango_vistas", "interacciones_totales",
    "tasa_interaccion", "tasa_imputada", "categoria_comercial", "n_senales_comerciales",
    "n_hashtags", "vistas_por_dia", "pais_agrupado",
    # marcas de calidad y de inclusión
    "f_fecha_incoherente", "f_sin_vistas", "f_likes_mayor_vistas", "f_duracion_inconsistente",
    "f_tipo_no_reconocido", "analisis_vistas", "analisis_tasa",
]
df_final = df[COLS_FINALES].copy()

# Redondeo: elimina el ruido de float32 y reduce el tamaño del CSV (no altera ninguna conclusión)
for c_, d_ in [("engagement_rate", 6), ("tasa_interaccion", 6), ("duracion_s", 3), ("vistas_por_dia", 2)]:
    df_final[c_] = df_final[c_].round(d_)
descartadas = [c for c in df_original.columns if c not in df_final.columns]
registrar("T5", "Selección de columnas pertinentes", "; ".join(descartadas), len(df_final),
          f"Se descartan {len(descartadas)} columnas de las {df_original.shape[1]} cargadas en AE2 porque no se usan en las preguntas "
          "(language, image_count, width, height, sound_id, product_id, seller_id, is_ai_generated, not_recommended_for_fyp, sound_muted, downloads), "
          "o se resumen en una derivada (stats_updated_at -> edad_dias) "
          "o fueron reemplazadas por una derivada (hashtags -> n_hashtags); tasa_interaccion_calc es intermedia y no se exporta. "
          "Se redondean las tasas a 6 decimales (ruido de float32), duracion_s a 3 y vistas_por_dia a 2. Se conservan todas las filas.")
print(f"df_final: {df_final.shape[0]:,} filas x {df_final.shape[1]} columnas")

[T5] Selección de columnas pertinentes: 200,000 registros afectados
df_final: 200,000 filas x 34 columnas


In [45]:
perfil_despues = perfil(df_final)

comparacion_resumen = pd.DataFrame([
    ("Filas", len(df_original), len(df_final)),
    ("Columnas", df_original.shape[1], df_final.shape[1]),
    ("Columnas numéricas", sum(familia_tipo(t) == "numérica" for t in df_original.dtypes), sum(familia_tipo(t) == "numérica" for t in df_final.dtypes)),
    ("Columnas de fecha", sum(familia_tipo(t) == "fecha" for t in df_original.dtypes), sum(familia_tipo(t) == "fecha" for t in df_final.dtypes)),
    ("Columnas booleanas", sum(familia_tipo(t) == "booleana" for t in df_original.dtypes), sum(familia_tipo(t) == "booleana" for t in df_final.dtypes)),
    ("Columnas de texto / lista", sum(familia_tipo(t) == "texto / lista / categoría" for t in df_original.dtypes), sum(familia_tipo(t) == "texto / lista / categoría" for t in df_final.dtypes)),
    ("Valores nulos (total)", int(df_original.isna().sum().sum()), int(df_final.isna().sum().sum())),
], columns=["indicador", "antes", "despues"])

comparacion_columnas = (perfil_antes.add_suffix("_antes")
                        .join(perfil_despues.add_suffix("_despues"), how="outer")
                        .rename_axis("columna").reset_index())
comparacion_columnas["estado"] = np.where(
    comparacion_columnas["tipo_despues"].isna(), "descartada",
    np.where(comparacion_columnas["tipo_antes"].isna(), "derivada", "conservada"))
display(comparacion_resumen)
comparacion_columnas

,indicador,antes,despues
0,Filas,200000,200000
1,Columnas,27,34
2,Columnas numéricas,20,16
3,Columnas de fecha,2,1
4,Columnas booleanas,0,10
5,Columnas de texto / lista,5,7
6,Valores nulos (total),220857,42699


,columna,tipo_antes,nulos_antes,tipo_despues,nulos_despues,estado
0,analisis_tasa,NaN,NaN,bool,0.00,derivada
1,analisis_vistas,NaN,NaN,bool,0.00,derivada
2,anio_publicacion,NaN,NaN,Int64,0.00,derivada
3,branded_content,object,0.00,string,0.00,conservada
4,categoria_comercial,NaN,NaN,object,0.00,derivada
5,comments,uint64,0.00,Int64,0.00,conservada
6,country,object,0.00,string,0.00,conservada
7,downloads,uint64,0.00,NaN,NaN,descartada
8,duracion_s,NaN,NaN,float64,"39,551.00",derivada
9,duration_ms,uint32,0.00,Int64,0.00,conservada


### 6.1 Diccionario de datos actualizado
Todas las columnas de `df_final` quedan documentadas; la celda falla si alguna no tiene definición.

In [46]:
DICC = {
 "video_id": ("Original", "Identificador de la publicación (clave del registro)", "Sin unidad"),
 "posted_at": ("Original", "Fecha y hora de publicación", "UTC"),
 "anio_publicacion": ("Derivada", "Año de publicación", "Año"),
 "edad_dias": ("Derivada", "Días entre la publicación y la medición de métricas (stats_updated_at); vacío si la fecha es incoherente", "Días"),
 "type": ("Original", "Tipo de publicación: video o photo (normalizado)", "—"),
 "duration_ms": ("Original", "Duración de la publicación; 0 en fotos según la ficha", "Milisegundos"),
 "country": ("Original", "Región asignada por TikTok; vacío reemplazado por 'SIN DATO'", "Código de país"),
 "is_ad": ("Original", "Indica publicidad (Spark Ads); nulo = TikTok no lo indicó", "Booleano"),
 "is_shop_video": ("Original", "Indica video de TikTok Shop; nulo = no indicado", "Booleano"),
 "branded_content": ("Original", "Contenido de marca (paid_partnership, shop_affiliate); vacío reemplazado por 'ninguno'", "—"),
 "views": ("Original", "Vistas al momento de stats_updated_at", "Vistas"),
 "likes": ("Original", "Me gusta al momento de stats_updated_at", "Conteo"),
 "comments": ("Original", "Comentarios al momento de stats_updated_at", "Conteo"),
 "shares": ("Original", "Compartidos al momento de stats_updated_at", "Conteo"),
 "saves": ("Original", "Guardados al momento de stats_updated_at", "Conteo"),
 "engagement_rate": ("Original", "Tasa de interacción publicada: (likes+comments+shares+saves)/views", "Proporción (0–1)"),
 "duracion_s": ("Derivada", "Duración en segundos (solo videos con duración > 0)", "Segundos"),
 "rango_duracion": ("Derivada", "Rango de duración del video; categorías propias para fotos y videos sin duración", "Rango"),
 "rango_vistas": ("Derivada", "Rango de alcance según vistas", "Rango"),
 "interacciones_totales": ("Derivada", "likes + comments + shares + saves", "Conteo"),
 "tasa_interaccion": ("Derivada", "engagement_rate; si es nulo, tasa recalculada", "Proporción (0–1)"),
 "tasa_imputada": ("Derivada", "True si la tasa se completó con la fórmula", "Booleano"),
 "categoria_comercial": ("Derivada", "Categoría comercial excluyente (publicidad > Shop > contenido de marca > ninguna)", "Categoría"),
 "n_senales_comerciales": ("Derivada", "Cantidad de señales comerciales presentes (0–3)", "Conteo"),
 "n_hashtags": ("Derivada", "Cantidad de hashtags de la publicación", "Conteo"),
 "vistas_por_dia": ("Derivada", "views / max(edad_dias, 1)", "Vistas por día"),
 "pais_agrupado": ("Derivada", "Países con más publicaciones; 'Otros países'; 'SIN DATO'", "Categoría"),
 "f_fecha_incoherente": ("Marca de calidad", "Fecha nula o publicación posterior a la medición", "Booleano"),
 "f_sin_vistas": ("Marca de calidad", "views = 0", "Booleano"),
 "f_likes_mayor_vistas": ("Marca de calidad", "likes > views con views > 0", "Booleano"),
 "f_duracion_inconsistente": ("Marca de calidad", "Foto con duración > 0 o video sin duración", "Booleano"),
 "f_tipo_no_reconocido": ("Marca de calidad", "type distinto de video o photo", "Booleano"),
 "analisis_vistas": ("Marca de calidad", "True si el registro entra en los cálculos de vistas", "Booleano"),
 "analisis_tasa": ("Marca de calidad", "True si el registro entra en los cálculos de tasa de interacción", "Booleano"),
}
faltan = [c for c in df_final.columns if c not in DICC]
sobran = [c for c in DICC if c not in df_final.columns]
assert not faltan and not sobran, f"Diccionario incompleto: faltan {faltan}, sobran {sobran}"

BOOL_01 = ["tasa_imputada", "f_fecha_incoherente", "f_sin_vistas", "f_likes_mayor_vistas",
           "f_duracion_inconsistente", "f_tipo_no_reconocido", "analisis_vistas", "analisis_tasa"]
diccionario = pd.DataFrame([
    {"variable": c, "origen": DICC[c][0], "significado": DICC[c][1],
     "tipo": str(df_final[c].dtype),
     "unidad_o_regla": DICC[c][2] + (" (0/1 en el CSV)" if c in BOOL_01 else "")} for c in df_final.columns])
print(f"Diccionario: {len(diccionario)} variables documentadas")
diccionario.head(10)

Diccionario: 34 variables documentadas


,variable,origen,significado,tipo,unidad_o_regla
0,video_id,Original,Identificador de la publicación (clave del reg...,uint64,Sin unidad
1,posted_at,Original,Fecha y hora de publicación,"datetime64[ms, UTC]",UTC
2,anio_publicacion,Derivada,Año de publicación,Int64,Año
3,edad_dias,Derivada,Días entre la publicación y la medición de mét...,Int64,Días
4,type,Original,Tipo de publicación: video o photo (normalizado),string,—
5,duration_ms,Original,Duración de la publicación; 0 en fotos según l...,Int64,Milisegundos
6,country,Original,Región asignada por TikTok; vacío reemplazado ...,string,Código de país
7,is_ad,Original,Indica publicidad (Spark Ads); nulo = TikTok n...,boolean,Booleano
8,is_shop_video,Original,Indica video de TikTok Shop; nulo = no indicado,boolean,Booleano
9,branded_content,Original,"Contenido de marca (paid_partnership, shop_aff...",string,—


## 7. Respuestas a las preguntas de negocio
Todas las tablas usan **los mismos registros**: `analisis_vistas` para vistas y `analisis_tasa` para la interacción. Las medidas centrales son **medianas y cuartiles**, porque las métricas están muy concentradas en pocas publicaciones; la *tasa agregada* es la suma de interacciones dividida por la suma de vistas del grupo. Todas las tasas se expresan en %.

In [47]:
base_v = df_final[df_final["analisis_vistas"]]
base_t = df_final[df_final["analisis_tasa"]]
N_BASE, N_TASA = len(base_v), len(base_t)
VISTAS_BASE = float(base_v["views"].astype("float64").sum())
print(f"Registros en el análisis de vistas: {N_BASE:,} de {len(df_final):,}")
print(f"Registros en el análisis de tasa de interacción: {N_TASA:,}")

def tabla_grupo(d, col, orden=None):
    """Agrupa por `col` y calcula alcance e interacción. Devuelve una tabla con `grupo` como texto."""
    b = d.loc[d["analisis_vistas"], [col, "views"]].copy()
    b["views"] = b["views"].astype("float64")
    g = b.groupby(col, dropna=False)["views"]
    t = pd.DataFrame({
        "n_publicaciones": g.size(),
        "vistas_total": g.sum(),
        "vistas_media": g.mean(),
        "vistas_mediana": g.median(),
        "vistas_p25": g.quantile(0.25),
        "vistas_p75": g.quantile(0.75),
    })
    r = d.loc[d["analisis_tasa"], [col, "tasa_interaccion", "interacciones_totales", "views"]].copy()
    r["interacciones_totales"] = r["interacciones_totales"].astype("float64")
    r["views"] = r["views"].astype("float64")
    gr = r.groupby(col, dropna=False)
    t["n_con_tasa"] = gr.size()
    t["tasa_mediana_pct"] = 100 * gr["tasa_interaccion"].median()
    t["tasa_agregada_pct"] = 100 * gr["interacciones_totales"].sum() / gr["views"].sum()
    t["n_con_tasa"] = t["n_con_tasa"].fillna(0).astype(int)
    t = t.reset_index().rename(columns={col: "grupo"})
    t["grupo"] = t["grupo"].astype(str)
    t["pct_publicaciones"] = 100 * t["n_publicaciones"] / t["n_publicaciones"].sum()
    t["pct_vistas"] = 100 * t["vistas_total"] / t["vistas_total"].sum()
    if isinstance(orden, list):
        presentes = [o for o in orden if o in set(t["grupo"])]
        resto = [x for x in t["grupo"] if x not in presentes]
        t = t.set_index("grupo").loc[presentes + resto].reset_index()
    elif orden == "grupo":
        t = t.sort_values("grupo")
    else:
        t = t.sort_values("n_publicaciones", ascending=False)
    t["n_publicaciones"] = t["n_publicaciones"].astype(int)
    t["vistas_total"] = t["vistas_total"].round().astype("int64")
    cols = ["grupo", "n_publicaciones", "pct_publicaciones", "vistas_total", "pct_vistas", "vistas_media",
            "vistas_mediana", "vistas_p25", "vistas_p75", "n_con_tasa", "tasa_mediana_pct", "tasa_agregada_pct"]
    return t[cols].reset_index(drop=True)

# ---------- Interpretación automática (descriptiva, sin causalidad) ----------
def extremos(t, col, ncol, etiqueta, fmt, excluir=()):
    s = t[~t["grupo"].isin(excluir)]
    s = s[(s[ncol] >= MIN_N_INTERP) & s[col].notna()]
    if len(s) < 2:
        return f"No hay al menos dos grupos con {MIN_N_INTERP} o más registros para comparar {etiqueta}."
    mx, mn = s.loc[s[col].idxmax()], s.loc[s[col].idxmin()]
    txt = (f"En {etiqueta}, el valor más alto corresponde a «{mx['grupo']}» ({fmt(mx[col])}) "
           f"y el más bajo a «{mn['grupo']}» ({fmt(mn[col])})")
    if mn[col] > 0:
        r = mx[col] / mn[col]
        txt += (f", es decir, {f_dec(r, 1)} veces el del grupo más bajo" if r >= 1.05
                else ", una diferencia menor al 5 %")
    return txt + "."

def composicion(t):
    a = t.loc[t["n_publicaciones"].idxmax()]
    return (f"El grupo más numeroso es «{a['grupo']}», con {f_pct(a['pct_publicaciones'])} de las publicaciones "
            f"y {f_pct(a['pct_vistas'])} de las vistas.")

def pequenos(t):
    p = t.loc[(t["n_publicaciones"] < MIN_N_INTERP), "grupo"].tolist()
    return (f"Los grupos con menos de {MIN_N_INTERP} registros ({', '.join(p)}) no se usan en las comparaciones."
            if p else "")

def interpretar(t, excluir=()):
    h_v = extremos(t, "vistas_mediana", "n_publicaciones", "la mediana de vistas", f_int, excluir)
    h_t = extremos(t, "tasa_mediana_pct", "n_con_tasa", "la tasa de interacción mediana", f_pct, excluir)
    h_a = extremos(t, "tasa_agregada_pct", "n_con_tasa", "la tasa de interacción agregada", f_pct, excluir)
    frases, hallazgos = [h_v, h_t, h_a], [h_v, h_t]
    if all(f.startswith("No hay") for f in frases):      # sin grupos comparables: un solo aviso
        aviso = (f"No hay al menos dos grupos con {MIN_N_INTERP} o más registros para comparar vistas o interacción entre grupos.")
        frases, hallazgos = [aviso], [aviso]
    texto = " ".join(x for x in [composicion(t)] + frases + [pequenos(t)] if x)
    return texto, hallazgos

INTERP, TABLAS = {}, {}
def mostrar(pid, titulo, t, texto, hallazgos, limitacion):
    INTERP[pid] = {"titulo": titulo, "texto": texto, "hallazgos": hallazgos, "limitacion": limitacion}
    display(t.round(2))
    print(textwrap.fill("Interpretación: " + texto, 130))
    print(textwrap.fill("Limitación: " + limitacion, 130))

Registros en el análisis de vistas: 200,000 de 200,000
Registros en el análisis de tasa de interacción: 183,783


### P1. Distribución general de vistas e interacción

In [48]:
def fila_stats(nombre, unidad, s):
    s = pd.Series(s).dropna().astype("float64")
    return {"variable": nombre, "unidad": unidad, "n": len(s), "media": s.mean(), "mediana": s.median(),
            "p25": s.quantile(0.25), "p75": s.quantile(0.75), "p95": s.quantile(0.95), "maximo": s.max()}

TABLAS["p1a_resumen_general"] = pd.DataFrame([
    fila_stats("views", "vistas", base_v["views"]),
    fila_stats("interacciones_totales", "interacciones", base_v["interacciones_totales"]),
    fila_stats("tasa_interaccion", "%", 100 * base_t["tasa_interaccion"]),
    fila_stats("duracion_s", "segundos (solo videos)", base_v["duracion_s"]),
    fila_stats("vistas_por_dia", "vistas por día", base_v["vistas_por_dia"]),
    fila_stats("n_hashtags", "hashtags", base_v["n_hashtags"]),
])
TABLAS["p1b_rango_vistas"] = tabla_grupo(df_final, "rango_vistas", orden=ORDEN_VISTAS + ["Sin dato"])

v = base_v["views"].astype("float64")
med, media = v.median(), v.mean()
k = max(1, int(round(0.01 * len(v))))
top1 = v.nlargest(k).sum() / v.sum() if v.sum() > 0 else float("nan")
tasa_p = 100 * base_t["tasa_interaccion"]
t1b = TABLAS["p1b_rango_vistas"]
alto = t1b[t1b["grupo"] == ORDEN_VISTAS[-1]]
txt_alto = (f" Las publicaciones con un millón de vistas o más son el {f_pct(alto['pct_publicaciones'].iloc[0])} "
            f"y reúnen el {f_pct(alto['pct_vistas'].iloc[0])} de las vistas.") if len(alto) else ""
texto = (f"La mediana de vistas por publicación es {f_int(med)}, mientras que la media es {f_int(media)}"
         + (f" ({f_dec(media / med, 1)} veces la mediana)" if med > 0 else "")
         + f"; el 1 % de publicaciones con más vistas ({f_int(k)}) concentra el {f_pct(100 * top1, 1)} del total."
         + txt_alto
         + f" La tasa de interacción mediana es {f_pct(tasa_p.median())} (rango intercuartil de {f_pct(tasa_p.quantile(.25))} a {f_pct(tasa_p.quantile(.75))}).")
mostrar("P1", "Distribución general de vistas e interacción", TABLAS["p1a_resumen_general"], texto,
        [texto.split(". ")[0] + "."],
        "Describe solo la muestra analizada; la distribución muy asimétrica hace que la media no represente a una publicación típica.")
TABLAS["p1b_rango_vistas"].round(2)

,variable,unidad,n,media,mediana,p25,p75,p95,maximo
0,views,vistas,200000,"1,770.94",41.00,20.00,87.00,375.00,"14,443,134.00"
1,interacciones_totales,interacciones,200000,"1,481.22",9.00,4.00,18.00,82.00,"1,926,910.00"
2,tasa_interaccion,%,183783,26.33,18.87,10.00,35.00,78.79,"1,228.57"
3,duracion_s,segundos (solo videos),160449,11.92,14.00,9.00,15.00,15.00,329.00
4,vistas_por_dia,vistas por día,200000,0.52,0.01,0.01,0.02,0.11,"3,998.65"
5,n_hashtags,hashtags,200000,0.64,0.00,0.00,0.00,4.00,17.00


Interpretación: La mediana de vistas por publicación es 41, mientras que la media es 1.771 (43,2 veces la mediana); el 1 % de
publicaciones con más vistas (2.000) concentra el 92,4 % del total. Las publicaciones con un millón de vistas o más son el 0,03 %
y reúnen el 25,63 % de las vistas. La tasa de interacción mediana es 18,87 % (rango intercuartil de 10,00 % a 35,00 %).
Limitación: Describe solo la muestra analizada; la distribución muy asimétrica hace que la media no represente a una publicación
típica.


,grupo,n_publicaciones,pct_publicaciones,vistas_total,pct_vistas,vistas_media,vistas_mediana,vistas_p25,vistas_p75,n_con_tasa,tasa_mediana_pct,tasa_agregada_pct
0,Menos de 1 mil,194840,97.42,14088014,3.98,72.31,40.00,19.00,81.00,179896,18.75,19.48
1,1 mil – 10 mil,2901,1.45,9588350,2.71,"3,305.19","2,356.00","1,432.00","4,500.00",1876,22.37,47.39
2,10 mil – 100 mil,1343,0.67,44160539,12.47,"32,882.01","25,501.00","15,902.00","42,111.50",1108,91.04,71.67
3,100 mil – 1 millón,860,0.43,195580535,55.22,"227,419.23","171,088.50","139,302.00","244,510.50",847,98.13,83.31
4,1 millón o más,56,0.03,90771347,25.63,"1,620,916.91","1,254,822.00","1,131,965.50","1,432,612.00",56,88.82,57.43


### P2. Según el tipo de publicación

In [49]:
TABLAS["p2_tipo_publicacion"] = tabla_grupo(df_final, "type")
t = TABLAS["p2_tipo_publicacion"]
texto, hall = interpretar(t)
mostrar("P2", "Según el tipo de publicación", t, texto, hall,
        "Las diferencias entre tipos pueden deberse a otros factores (antigüedad, país, uso comercial); no demuestran que el formato cause el resultado.")

,grupo,n_publicaciones,pct_publicaciones,vistas_total,pct_vistas,vistas_media,vistas_mediana,vistas_p25,vistas_p75,n_con_tasa,tasa_mediana_pct,tasa_agregada_pct
0,video,160456,80.23,338279399,95.51,"2,108.24",42.00,20.00,88.00,147477,18.75,72.56
1,photo,39544,19.77,15909386,4.49,402.32,39.00,19.00,80.00,36306,19.05,58.27


Interpretación: El grupo más numeroso es «video», con 80,23 % de las publicaciones y 95,51 % de las vistas. En la mediana de
vistas, el valor más alto corresponde a «video» (42) y el más bajo a «photo» (39), es decir, 1,1 veces el del grupo más bajo. En
la tasa de interacción mediana, el valor más alto corresponde a «photo» (19,05 %) y el más bajo a «video» (18,75 %), una
diferencia menor al 5 %. En la tasa de interacción agregada, el valor más alto corresponde a «video» (72,56 %) y el más bajo a
«photo» (58,27 %), es decir, 1,2 veces el del grupo más bajo.
Limitación: Las diferencias entre tipos pueden deberse a otros factores (antigüedad, país, uso comercial); no demuestran que el
formato cause el resultado.


### P3. Según el rango de duración

In [50]:
TABLAS["p3_rango_duracion"] = tabla_grupo(df_final, "rango_duracion", orden=ORDEN_DURACION)
t = TABLAS["p3_rango_duracion"]
especiales = ORDEN_DURACION[5:] + ["Tipo no reconocido"]
texto, hall = interpretar(t, excluir=especiales)
texto += " Las fotos y los videos sin duración registrada se muestran en la tabla pero no entran en la comparación de rangos."
mostrar("P3", "Según el rango de duración", t, texto, hall,
        "La duración está redondeada al segundo en videos antiguos y los rangos son una decisión del equipo; la comparación es descriptiva.")

,grupo,n_publicaciones,pct_publicaciones,vistas_total,pct_vistas,vistas_media,vistas_mediana,vistas_p25,vistas_p75,n_con_tasa,tasa_mediana_pct,tasa_agregada_pct
0,0–15 s,159365,79.68,336316755,94.95,"2,110.36",42.00,20.00,88.00,146531,18.75,72.65
1,15–30 s,685,0.34,1184825,0.33,"1,729.67",55.00,26.00,112.00,587,23.81,62.75
2,30–60 s,393,0.20,429870,0.12,"1,093.82",51.00,25.00,117.00,351,26.19,81.32
3,1–3 min,4,0.00,178,0.00,44.50,18.50,16.00,47.00,3,52.63,30.36
4,Más de 3 min,2,0.00,121,0.00,60.50,60.50,46.75,74.25,2,32.01,31.40
5,Video sin duración registrada,7,0.00,347650,0.10,"49,664.29","1,552.00","1,179.00","57,010.00",3,16.14,14.70
6,Foto (sin duración),39544,19.77,15909386,4.49,402.32,39.00,19.00,80.00,36306,19.05,58.27


Interpretación: El grupo más numeroso es «0–15 s», con 79,68 % de las publicaciones y 94,95 % de las vistas. En la mediana de
vistas, el valor más alto corresponde a «15–30 s» (55) y el más bajo a «0–15 s» (42), es decir, 1,3 veces el del grupo más bajo.
En la tasa de interacción mediana, el valor más alto corresponde a «30–60 s» (26,19 %) y el más bajo a «0–15 s» (18,75 %), es
decir, 1,4 veces el del grupo más bajo. En la tasa de interacción agregada, el valor más alto corresponde a «30–60 s» (81,32 %) y
el más bajo a «15–30 s» (62,75 %), es decir, 1,3 veces el del grupo más bajo. Los grupos con menos de 100 registros (1–3 min, Más
de 3 min, Video sin duración registrada) no se usan en las comparaciones. Las fotos y los videos sin duración registrada se
muestran en la tabla pero no entran en la comparación de rangos.
Limitación: La duración está redondeada al segundo en videos antiguos y los rangos son una decisión del equipo; la comparación es
descriptiva.


### P4. Según el país

In [51]:
TABLAS["p4_pais"] = tabla_grupo(df_final, "pais_agrupado", orden=top_paises + ["Otros países", "SIN DATO"])
TABLAS["p4b_pais_completo"] = tabla_grupo(df_final, "country")
t = TABLAS["p4_pais"]
texto, hall = interpretar(t, excluir=["Otros países", "SIN DATO"])
sd = t[t["grupo"] == "SIN DATO"]
if len(sd):
    texto += f" El {f_pct(sd['pct_publicaciones'].iloc[0])} de las publicaciones no tiene país asignado (SIN DATO) y no puede compararse."
mostrar("P4", "Según el país", t, texto, hall,
        "El país es una etiqueta asignada por TikTok, puede estar vacía y la muestra no es representativa de cada país.")

,grupo,n_publicaciones,pct_publicaciones,vistas_total,pct_vistas,vistas_media,vistas_mediana,vistas_p25,vistas_p75,n_con_tasa,tasa_mediana_pct,tasa_agregada_pct
0,SIN DATO,200000,100.00,354188785,100.00,"1,770.94",41.00,20.00,87.00,183783,18.87,71.96


Interpretación: El grupo más numeroso es «SIN DATO», con 100,00 % de las publicaciones y 100,00 % de las vistas. No hay al menos
dos grupos con 100 o más registros para comparar vistas o interacción entre grupos. El 100,00 % de las publicaciones no tiene país
asignado (SIN DATO) y no puede compararse.
Limitación: El país es una etiqueta asignada por TikTok, puede estar vacía y la muestra no es representativa de cada país.


### P5. Según el carácter comercial

In [52]:
TABLAS["p5a_categoria_comercial"] = tabla_grupo(df_final, "categoria_comercial",
                                                orden=[CAT_AD, CAT_SHOP, CAT_MARCA, CAT_NINGUNA])
tmp = df_final.copy()
tmp["_ad"] = np.where(df_final["is_ad"].fillna(False).astype(bool), "Sí", "No")
tmp["_shop"] = np.where(df_final["is_shop_video"].fillna(False).astype(bool), "Sí", "No")
tmp["_marca"] = np.where(df["branded_content"].to_numpy() != "ninguno", "Sí", "No")
partes = []
for col, nombre in [("_ad", "Publicidad (is_ad)"), ("_shop", "TikTok Shop (is_shop_video)"), ("_marca", "Contenido de marca (branded_content)")]:
    p = tabla_grupo(tmp, col, orden=["Sí", "No"])
    p.insert(0, "indicador", nombre)
    partes.append(p)
TABLAS["p5b_indicadores_comerciales"] = pd.concat(partes, ignore_index=True)

t = TABLAS["p5a_categoria_comercial"]
texto, hall = interpretar(t)
solap = int((df_final["n_senales_comerciales"] >= 2).sum())
texto += (f" {f_int(solap)} publicaciones ({f_pct(100 * solap / len(df_final))}) tienen dos o más señales comerciales y se asignan "
          "a una sola categoría por prioridad; la tabla por indicador compara «Sí» y «No» para cada señal por separado.")
mostrar("P5", "Según el carácter comercial", t, texto, hall,
        "Sin datos de inversión, costos ni conversiones no puede afirmarse qué formato comercial es más rentable; solo se describe alcance e interacción.")
display(TABLAS["p5b_indicadores_comerciales"].round(2))

,grupo,n_publicaciones,pct_publicaciones,vistas_total,pct_vistas,vistas_media,vistas_mediana,vistas_p25,vistas_p75,n_con_tasa,tasa_mediana_pct,tasa_agregada_pct
0,Sin señal comercial,200000,100.00,354188785,100.00,"1,770.94",41.00,20.00,87.00,183783,18.87,71.96


Interpretación: El grupo más numeroso es «Sin señal comercial», con 100,00 % de las publicaciones y 100,00 % de las vistas. No hay
al menos dos grupos con 100 o más registros para comparar vistas o interacción entre grupos. 0 publicaciones (0,00 %) tienen dos o
más señales comerciales y se asignan a una sola categoría por prioridad; la tabla por indicador compara «Sí» y «No» para cada
señal por separado.
Limitación: Sin datos de inversión, costos ni conversiones no puede afirmarse qué formato comercial es más rentable; solo se
describe alcance e interacción.


,indicador,grupo,n_publicaciones,pct_publicaciones,vistas_total,pct_vistas,vistas_media,vistas_mediana,vistas_p25,vistas_p75,n_con_tasa,tasa_mediana_pct,tasa_agregada_pct
0,Publicidad (is_ad),No,200000,100.00,354188785,100.00,"1,770.94",41.00,20.00,87.00,183783,18.87,71.96
1,TikTok Shop (is_shop_video),No,200000,100.00,354188785,100.00,"1,770.94",41.00,20.00,87.00,183783,18.87,71.96
2,Contenido de marca (branded_content),No,200000,100.00,354188785,100.00,"1,770.94",41.00,20.00,87.00,183783,18.87,71.96


### P6. Según el año de publicación (contexto)

In [53]:
TABLAS["p6_anio_publicacion"] = tabla_grupo(df_final, "anio_publicacion", orden="grupo")
ctx = (base_v.assign(grupo=base_v["anio_publicacion"].astype(str))
       .groupby("grupo")
       .agg(edad_dias_mediana=("edad_dias", lambda s: s.astype("float64").median()),
            vistas_por_dia_mediana=("vistas_por_dia", "median"))
       .reset_index())
TABLAS["p6_anio_publicacion"] = TABLAS["p6_anio_publicacion"].merge(ctx, on="grupo", how="left")
t = TABLAS["p6_anio_publicacion"]
texto, hall = interpretar(t)
texto += (f" Periodo cubierto por la muestra: {int(df_final['anio_publicacion'].min())} a {int(df_final['anio_publicacion'].max())}."
          " Las publicaciones más antiguas llevan más tiempo en línea al medir las métricas; por eso se incluyen la edad mediana y las vistas por día.")
mostrar("P6", "Según el año de publicación", t, texto, hall,
        "Cada registro se midió en una fecha distinta y la muestra proviene de un archivo concreto; los años no son una serie temporal completa de TikTok.")

,grupo,n_publicaciones,pct_publicaciones,vistas_total,pct_vistas,vistas_media,vistas_mediana,vistas_p25,vistas_p75,n_con_tasa,tasa_mediana_pct,tasa_agregada_pct,edad_dias_mediana,vistas_por_dia_mediana
0,2016,200000,100.00,354188785,100.00,"1,770.94",41.00,20.00,87.00,183783,18.87,71.96,3508,0.01


Interpretación: El grupo más numeroso es «2016», con 100,00 % de las publicaciones y 100,00 % de las vistas. No hay al menos dos
grupos con 100 o más registros para comparar vistas o interacción entre grupos. Periodo cubierto por la muestra: 2016 a 2016. Las
publicaciones más antiguas llevan más tiempo en línea al medir las métricas; por eso se incluyen la edad mediana y las vistas por
día.
Limitación: Cada registro se midió en una fecha distinta y la muestra proviene de un archivo concreto; los años no son una serie
temporal completa de TikTok.


## 8. Verificación de consistencia
Se comprueba que en cada tabla los registros, los porcentajes y el total de vistas cuadren con la base de análisis.

In [54]:
verificaciones = []
def verificar(nombre, esperado, obtenido, ok):
    verificaciones.append({"verificacion": nombre, "esperado": esperado, "obtenido": obtenido, "resultado": "OK" if ok else "REVISAR"})

def verificar_tabla(nombre, t):
    verificar(f"{nombre}: suma de publicaciones", f"{N_BASE:,}", f"{int(t['n_publicaciones'].sum()):,}", int(t["n_publicaciones"].sum()) == N_BASE)
    verificar(f"{nombre}: suma de % de publicaciones", "100", f"{t['pct_publicaciones'].sum():.6f}", abs(t["pct_publicaciones"].sum() - 100) < 1e-6)
    verificar(f"{nombre}: suma de vistas", f"{VISTAS_BASE:,.0f}", f"{t['vistas_total'].sum():,.0f}", abs(t["vistas_total"].sum() - VISTAS_BASE) <= 1)
    verificar(f"{nombre}: registros con tasa", f"{N_TASA:,}", f"{int(t['n_con_tasa'].sum()):,}", int(t["n_con_tasa"].sum()) == N_TASA)

for nombre in ["p1b_rango_vistas", "p2_tipo_publicacion", "p3_rango_duracion", "p4_pais", "p4b_pais_completo", "p5a_categoria_comercial", "p6_anio_publicacion"]:
    verificar_tabla(nombre, TABLAS[nombre])
for ind, sub in TABLAS["p5b_indicadores_comerciales"].groupby("indicador"):
    verificar_tabla(f"p5b ({ind})", sub)

verificar("Registros finales = registros cargados - duplicados", f"{len(df_original) - n_dup_id:,}", f"{len(df_final):,}", len(df_final) == len(df_original) - n_dup_id)
verificar("analisis_tasa incluido en analisis_vistas", "True", str(bool((~df_final["analisis_tasa"] | df_final["analisis_vistas"]).all())), bool((~df_final["analisis_tasa"] | df_final["analisis_vistas"]).all()))
tabla_verif = pd.DataFrame(verificaciones)
assert (tabla_verif["resultado"] == "OK").all(), "Hay verificaciones de consistencia que no cuadran"
print(f"{len(tabla_verif)} verificaciones: todas OK")
tabla_verif

42 verificaciones: todas OK


,verificacion,esperado,obtenido,resultado
0,p1b_rango_vistas: suma de publicaciones,"200,000","200,000",OK
1,p1b_rango_vistas: suma de % de publicaciones,100,100.000000,OK
2,p1b_rango_vistas: suma de vistas,"354,188,785","354,188,785",OK
3,p1b_rango_vistas: registros con tasa,"183,783","183,783",OK
4,p2_tipo_publicacion: suma de publicaciones,"200,000","200,000",OK
5,p2_tipo_publicacion: suma de % de publicaciones,100,100.000000,OK
6,p2_tipo_publicacion: suma de vistas,"354,188,785","354,188,785",OK
7,p2_tipo_publicacion: registros con tasa,"183,783","183,783",OK
8,p3_rango_duracion: suma de publicaciones,"200,000","200,000",OK
9,p3_rango_duracion: suma de % de publicaciones,100,100.000000,OK


## 9. Exportación de datos preparados, tablas y calidad (.csv)

In [55]:
# Datos preparados y diccionario
exportable = df_final.copy()
exportable[BOOL_01] = exportable[BOOL_01].astype(int)      # 0/1 reduce el tamaño del archivo
exportable.to_csv(RUTA_PREPARADO / "tiktok_preparado.csv", index=False, encoding="utf-8",
                  date_format="%Y-%m-%d %H:%M:%S")
diccionario.to_csv(RUTA_PREPARADO / "diccionario_actualizado.csv", index=False, encoding="utf-8")

# Tablas de resultados (una por pregunta)
for nombre, t in TABLAS.items():
    t.to_csv(RUTA_TABLAS / f"{nombre}.csv", index=False, encoding="utf-8")

# Evidencias de calidad
pd.DataFrame(bitacora).to_csv(RUTA_CALIDAD / "bitacora_transformaciones.csv", index=False, encoding="utf-8")
tabla_controles.to_csv(RUTA_CALIDAD / "controles_calidad.csv", index=False, encoding="utf-8")
pd.DataFrame(valid).to_csv(RUTA_CALIDAD / "validaciones_enriquecimiento.csv", index=False, encoding="utf-8")
comparacion_resumen.to_csv(RUTA_CALIDAD / "comparacion_resumen.csv", index=False, encoding="utf-8")
comparacion_columnas.to_csv(RUTA_CALIDAD / "comparacion_columnas.csv", index=False, encoding="utf-8")
pd.DataFrame(derivadas).to_csv(RUTA_CALIDAD / "variables_derivadas.csv", index=False, encoding="utf-8")
tabla_verif.to_csv(RUTA_CALIDAD / "verificaciones_consistencia.csv", index=False, encoding="utf-8")
print("Archivos exportados:")
for ruta in sorted(Path("datos").rglob("*.csv")):
    print(f"  {ruta}  ({ruta.stat().st_size / 1024:,.0f} KB)")

Archivos exportados:
  datos/calidad/bitacora_transformaciones.csv  (2 KB)
  datos/calidad/comparacion_columnas.csv  (2 KB)
  datos/calidad/comparacion_resumen.csv  (0 KB)
  datos/calidad/controles_calidad.csv  (1 KB)
  datos/calidad/validaciones_enriquecimiento.csv  (1 KB)
  datos/calidad/variables_derivadas.csv  (2 KB)
  datos/calidad/verificaciones_consistencia.csv  (3 KB)
  datos/preparado/diccionario_actualizado.csv  (3 KB)
  datos/preparado/tiktok_preparado.csv  (39,953 KB)
  datos/tablas/p1a_resumen_general.csv  (0 KB)
  datos/tablas/p1b_rango_vistas.csv  (1 KB)
  datos/tablas/p2_tipo_publicacion.csv  (0 KB)
  datos/tablas/p3_rango_duracion.csv  (1 KB)
  datos/tablas/p4_pais.csv  (0 KB)
  datos/tablas/p4b_pais_completo.csv  (0 KB)
  datos/tablas/p5a_categoria_comercial.csv  (0 KB)
  datos/tablas/p5b_indicadores_comerciales.csv  (1 KB)
  datos/tablas/p6_anio_publicacion.csv  (0 KB)


In [56]:
# Comprobar que lo exportado coincide con lo calculado en el notebook
chk = pd.read_csv(RUTA_PREPARADO / "tiktok_preparado.csv", usecols=["video_id", "views", "interacciones_totales"])
assert len(chk) == len(df_final), "El CSV preparado no tiene las mismas filas"
assert int(chk["views"].sum()) == int(df_final["views"].sum()), "Las vistas del CSV no coinciden"
assert int(chk["interacciones_totales"].sum()) == int(df_final["interacciones_totales"].sum())
assert int(pd.read_csv(RUTA_PREPARADO / "tiktok_preparado.csv", usecols=["analisis_tasa"])["analisis_tasa"].sum()) == N_TASA

for nombre, t in TABLAS.items():
    leida = pd.read_csv(RUTA_TABLAS / f"{nombre}.csv", dtype={c: str for c in ("grupo", "indicador") if c in t.columns})
    pd.testing.assert_frame_equal(leida.reset_index(drop=True), t.reset_index(drop=True),
                                  check_dtype=False, check_exact=False, rtol=1e-9)
print(f"OK: el CSV preparado ({len(chk):,} filas) y las {len(TABLAS)} tablas exportadas coinciden con el notebook")

OK: el CSV preparado (200,000 filas) y las 9 tablas exportadas coinciden con el notebook


## 10. Resumen descriptivo (.txt) del DataFrame final
Se genera con la clase `DatasetSummarizer` (trabajada en Triple A) a partir de `df_final`, y se regenera en cada ejecución.

In [57]:
resumen = DatasetSummarizer(output_dir=RUTA_RESULTADOS, filename="resumen_dataframe.txt")
ruta_txt = resumen.summarize(
    df_final,
    source=f"{DATASET_ID} (split: {loader.split_used}) – AE3, DataFrame transformado y enriquecido",
    description=(f"corrección de tipos, normalización de categorías, marcado de calidad, {len(derivadas)} grupos de variables "
                 f"derivadas y selección de {df_final.shape[1]} columnas (ver datos/calidad/bitacora_transformaciones.csv)"),
    metadata={
        "Enlace": ENLACE,
        "Licencia (según la ficha)": "CC BY-NC 4.0",
        "Revisión (commit)": loader.revision,
        "Fecha de consulta": FECHA_CONSULTA,
        "Filas del dataset completo (ficha)": f"{TOTAL_FILAS_DATASET:,}",
        "¿Muestra?": ("Sí. " + loader.sample_note) if loader.is_sample else "No",
        "Registros que entran al análisis de vistas": f"{N_BASE:,}",
        "Registros que entran al análisis de tasa": f"{N_TASA:,}",
    })
print("Archivo generado:", ruta_txt.resolve())

Archivo generado: /content/repo/AE3/repo/AE3/repo/AE3/resultados/resumen_dataframe.txt


In [58]:
# El .txt debe corresponder al DataFrame final de ESTA ejecución
texto_txt = ruta_txt.read_text(encoding="utf-8")
assert f"- Filas: {len(df_final)}" in texto_txt
assert f"- Columnas: {df_final.shape[1]}" in texto_txt
nulos_final = df_final.isna().sum()
for col in df_final.columns:
    linea = [l for l in texto_txt.splitlines() if l.startswith(f"{col} ")]
    assert linea, f"La columna {col} no aparece en el resumen"
    assert linea[0].rstrip().endswith(f"| {int(nulos_final[col])}"), f"Nulos distintos para {col}"
print("OK: el resumen .txt coincide con df_final (filas, columnas y nulos)\n")
print(texto_txt)

OK: el resumen .txt coincide con df_final (filas, columnas y nulos)

RESUMEN DEL DATAFRAME
Generado el: 2026-11-08 17:42:54

0. ORIGEN
- Origen: datasocial/tiktok-5.6B-videos (split: train) – AE3, DataFrame transformado y enriquecido
- Enlace: https://huggingface.co/datasets/datasocial/tiktok-5.6B-videos
- Licencia (según la ficha): CC BY-NC 4.0
- Revisión (commit): a466d03349b4afa490ac8906c8b5d17f6006abc0
- Fecha de consulta: 2026-11-08
- Filas del dataset completo (ficha): 5,598,141,717
- ¿Muestra?: Sí. Archivo 'data/2016-11.parquet' elegido al azar (semilla 42) entre 148 archivos parquet; 1 grupos de filas repartidos de 1; hasta 200,000 filas. No es una muestra aleatoria simple del dataset completo.
- Registros que entran al análisis de vistas: 200,000
- Registros que entran al análisis de tasa: 183,783
- Transformación aplicada: corrección de tipos, normalización de categorías, marcado de calidad, 15 grupos de variables derivadas y selección de 34 columnas (ver datos/calidad/bitaco

## 11. Metadatos de la ejecución e interpretaciones (insumo del documento)

In [59]:
meta = {
    "dataset_id": DATASET_ID, "enlace": ENLACE, "split": loader.split_used, "revision": loader.revision,
    "fecha_consulta": FECHA_CONSULTA, "fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "licencia": "CC BY-NC 4.0", "total_filas_dataset": TOTAL_FILAS_DATASET,
    "es_muestra": bool(loader.is_sample), "nota_muestra": loader.sample_note, "n_filas_pedidas": N_FILAS, "semilla": SEED,
    "n_filas_cargadas": int(len(df_original)), "n_cols_cargadas": int(df_original.shape[1]),
    "n_nulos_cargados": int(df_original.isna().sum().sum()),
    "n_filas_final": int(len(df_final)), "n_cols_final": int(df_final.shape[1]),
    "n_nulos_final": int(df_final.isna().sum().sum()),
    "n_analisis_vistas": int(N_BASE), "n_analisis_tasa": int(N_TASA),
    "n_duplicados_video_id": int(n_dup_id), "n_tasas_imputadas": int(df_final["tasa_imputada"].sum()),
    "n_paises_sin_dato": int(paises_vacios), "n_marca_ninguno": int(marcas_vacias),
    "periodo_min": str(base_v["posted_at"].min()), "periodo_max": str(base_v["posted_at"].max()),
    "anio_min": int(base_v["anio_publicacion"].min()), "anio_max": int(base_v["anio_publicacion"].max()),
    "top_paises": top_paises, "min_n_interp": MIN_N_INTERP, "tolerancia_tasa": TOL_TASA,
    "vistas_total_base": VISTAS_BASE, "n_derivadas": len(derivadas),
    "columnas_descartadas": descartadas,
    "tablas": sorted(TABLAS), "n_verificaciones": int(len(tabla_verif)),
}
(RUTA_RESULTADOS / "metadatos_ejecucion.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
(RUTA_RESULTADOS / "interpretaciones.json").write_text(json.dumps(INTERP, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: meta[k] for k in ["n_filas_cargadas", "n_filas_final", "n_cols_final", "n_analisis_vistas", "n_analisis_tasa", "periodo_min", "periodo_max"]}, ensure_ascii=False, indent=2))

{
  "n_filas_cargadas": 200000,
  "n_filas_final": 200000,
  "n_cols_final": 34,
  "n_analisis_vistas": 200000,
  "n_analisis_tasa": 183783,
  "periodo_min": "2016-11-01 00:00:01+00:00",
  "periodo_max": "2016-11-16 22:27:51+00:00"
}


## 12. Documento de avance
La última celda actualiza el Word de AE2 con las secciones de AE3 (metodología, diccionario, resultados, conclusiones) **a partir de los archivos exportados**, de modo que el documento coincide con el notebook. Requiere `python-docx` (ver `requirements.txt`).

In [60]:
try:
    import generar_documento
    ruta_doc = generar_documento.main()
    print("Documento generado:", ruta_doc)
except ImportError as e:
    print("No se generó el documento (falta una dependencia):", e)
except FileNotFoundError as e:
    print("No se generó el documento:", e)

Documento generado: documento/Documento_AE3_Avance.docx


## 13. Conclusiones del avance
- **Qué permite conocer:** las tablas P1–P6 describen, para la muestra, cómo se distribuyen las vistas y la tasa de interacción según tipo, duración, país, carácter comercial y año. Las interpretaciones impresas arriba (y en `resultados/interpretaciones.json`) resumen los hallazgos con sus cifras.
- **Qué queda sin resolver:** causalidad (¿el formato *produce* más interacción?), rentabilidad (no hay costos ni conversiones), representatividad (la muestra proviene de un archivo concreto) y el efecto de la antigüedad (solo se contextualiza con P6).
- **Para AE4:** las tablas de `datos/tablas/` están listas para graficar (barras por grupo, distribución por rango de vistas y evolución por año); el dataset preparado está en `datos/preparado/tiktok_preparado.csv`.

In [61]:
!zip -qr resultados_AE3.zip datos resultados documento
from google.colab import files
files.download("resultados_AE3.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>